In [ ]:
pre code { overflow-x: scroll; white-space: pre; }

# Overview

This document characterizes the operating characteristics of the screening pipeline
(Steps 1-3 of the main manuscript) across four causal scenarios, organized into two
parts:

- **Part 1 (Power):** Scenario A — the canonical downstream-mediation scenario in
  which the GxM null is false. We sweep the true `beta_GxM` and cross it with
  measurement error on the exposure E and on the metabolite M, to characterize when
  the GxM test is more powerful than the direct GxE test.
- **Part 2 (Type I error):** Scenarios B, C, and D — non-mediating scenarios in which
  the GxM null on Y is true but other structural features (upstream bioaccumulation,
  reverse causation, confounding) may inflate the Step-3 rejection rate. We fix one
  strong parameter setting per scenario and report the empirical Type I error rate of
  the GxM test. For Scenarios B and C we additionally show a practical `G:E`-adjusted
  test and a mechanistic check that removes the marginal E-M correlation; for Scenario
  D we compare adjustments for U, `G x U`, and `G x E`, with the confounder U itself
  measured with error.

The four causal scenarios:

- **A. Downstream signaling** (canonical mediator): `E -> M -> Y`, with `G x M -> Y`.
- **B. Upstream bioaccumulation**: `G x E -> M`, then `M -> Y`. Motivating GxE on Y is
  real, but no `G x M` on Y.
- **C. Reverse causation**: `G x E -> Y`, then `Y -> M`. M is downstream of Y.
- **D. Confounded null**: A common cause `U` is shared between E and M (measured only
  with error in practice); M has no causal role on Y.

Dashed red arrows denote a moderation effect (the head node's relationship to its
target depends on the modifier); solid black arrows denote main effects. Greek labels
match the parameter names used in the data-generating mechanisms below.

In [ ]:
library(tidyverse)

nodes_list <- list(
  A = tibble(node = c("G","E","M","Y"),
             x = c( 1.0, 0, 1, 2), y = c(1.2, 0, 0, 0)),
  B = tibble(node = c("G","E","M","Y"),
             x = c( 0.5, 0, 1, 2), y = c(1.2, 0, 0, 0)),
  C = tibble(node = c("G","E","Y","M"),
             x = c( 0.5, 0, 1, 2), y = c(1.2, 0, 0, 0)),
  D = tibble(node = c("G","U","E","M","Y"),
             x = c( 0.45, 1, 0, 1.4, 2), y = c(1.2, 1.2, 0, 0.5, 0))
)
# For mod edges, `to` is "FROM->TO" naming the main edge that's being modified;
# the arrow will land at the midpoint of that edge. For main edges, `to` is a node.
# `label`     : Greek coefficient name (used by the clean Fig-1 DAGs).
# `label_val` : name = default value (used by the annotated parameter-reference
#               supplement); `nx`/`ny` nudge those annotated labels only.
edges_list <- list(
  A = tibble(from = c("E","M","G"),     to = c("M","Y","M->Y"),
             label = c("beta[EM]","beta[MY]","beta[GxM]"),
             label_val = c("beta[EM]==0.3","beta[MY]==0.5","beta[GxM]"),
             nx = c(0, 0, 0), ny = c(0, 0, 0),
             kind  = c("main","main","mod")),
  B = tibble(from = c("E","G","M"),     to = c("M","E->M","Y"),
             label = c("beta[EM]","beta[GxE]","beta[MY]"),
             label_val = c("beta[EM]==0.3","beta[GxE]==0.4","beta[MY]==0.5"),
             nx = c(0, 0.10, 0), ny = c(0, 0, 0),
             kind  = c("main","mod","main")),
  C = tibble(from = c("E","G","Y"),     to = c("Y","E->Y","M"),
             label = c("beta[EY]","beta[GxE]","beta[YM]"),
             label_val = c("beta[EY]==0.3","beta[GxE]==0.2","beta[YM]==0.5"),
             nx = c(0, 0.10, 0), ny = c(0, 0, 0),
             kind  = c("main","mod","main")),
  D = tibble(from = c("U","U","E","G"), to = c("E","M","Y","E->Y"),
             label = c("beta[UE]","beta[UM]","beta[EY]","beta[GxE]"),
             label_val = c("beta[UE]==0.8","beta[UM]==0.7","beta[EY]==0.3","beta[GxE]==0.2"),
             nx = c(-0.06, 0.12, 0, 0.12), ny = c(-0.10, 0, 0, 0.02),
             kind  = c("main","main","main","mod"))
)

resolve_endpoint <- function(to_str, pos) {
  if (grepl("->", to_str, fixed = TRUE)) {
    parts <- strsplit(to_str, "->", fixed = TRUE)[[1]]
    x1 <- pos$x[match(parts[1], pos$node)]; y1 <- pos$y[match(parts[1], pos$node)]
    x2 <- pos$x[match(parts[2], pos$node)]; y2 <- pos$y[match(parts[2], pos$node)]
    c((x1 + x2) / 2, (y1 + y2) / 2)
  } else {
    c(pos$x[match(to_str, pos$node)], pos$y[match(to_str, pos$node)])
  }
}

scenarios <- names(nodes_list)
nodes_all <- map(scenarios, \(s) mutate(nodes_list[[s]], scenario = s)) |> bind_rows()
edges_all <- map(scenarios, \(s) {
  df  <- edges_list[[s]]
  pos <- nodes_list[[s]]
  df <- df |>
    mutate(
      x         = pos$x[match(from, pos$node)],
      y         = pos$y[match(from, pos$node)],
      endpoints = map(to, ~ resolve_endpoint(.x, pos)),
      xend      = map_dbl(endpoints, 1),
      yend      = map_dbl(endpoints, 2)
    ) |>
    select(-endpoints)
  # Shorten each edge so the arrowhead doesn't land inside the node label.
  pad <- 0.20
  df <- df |>
    mutate(
      dx = xend - x,  dy = yend - y,
      L  = sqrt(dx^2 + dy^2),
      x  = x    + (pad / L) * dx,
      y  = y    + (pad / L) * dy,
      xend = xend - (pad / L) * dx,
      yend = yend - (pad / L) * dy
    )
  # Coefficient label position: perpendicular offset from arrow midpoint. For
  # horizontal arrows (dy ~ 0), force the label below the arrow to avoid
  # collision with mod arrows that land on the line from above.
  off <- 0.14
  df |>
    mutate(
      perp_x = if_else(abs(dy) < 1e-6,  0, -dy / L),
      perp_y = if_else(abs(dy) < 1e-6, -1,  dx / L),
      label_x = (x + xend) / 2 + off * perp_x,
      label_y = (y + yend) / 2 + off * perp_y,
      scenario = s
    ) |>
    select(-dx, -dy, -L, -perp_x, -perp_y)
}) |> bind_rows()

scenario_labels <- c(A = "Molecular mediator (primary)",
                     B = "Upstream GxE",
                     C = "Reverse causation",
                     D = "E-M Confounding")

build_dag_plot <- function(scenarios_to_show) {
  edges_subset <- edges_all |> filter(scenario %in% scenarios_to_show)
  nodes_subset <- nodes_all |> filter(scenario %in% scenarios_to_show)
  ggplot() +
    geom_segment(data = edges_subset,
                 aes(x = x, y = y, xend = xend, yend = yend,
                     linetype = kind, color = kind),
                 arrow = arrow(length = unit(0.10, "in"), type = "closed"),
                 linewidth = 0.55) +
    geom_text(data = edges_subset,
              aes(x = label_x, y = label_y, label = label, color = kind),
              parse = TRUE, size = 3.7, show.legend = FALSE) +
    geom_label(data = nodes_subset,
               aes(x = x, y = y, label = node),
               size = 4.6, fontface = "bold", fill = "white",
               label.padding = unit(0.22, "lines")) +
    facet_wrap(~ scenario, nrow = 1,
               labeller = labeller(scenario = scenario_labels)) +
    scale_linetype_manual(values = c(main = "solid", mod = "dashed")) +
    scale_color_manual(values    = c(main = "grey20", mod = "firebrick")) +
    coord_cartesian(xlim = c(-0.2, 2.2), ylim = c(-0.4, 1.5), clip = "off",
                    expand = FALSE) +
    theme_void(base_size = 13) +
    theme(legend.position = "none",
          strip.text      = element_text(face = "bold"),
          panel.spacing   = unit(0.2, "lines"),
          plot.margin     = margin(2, 2, 2, 2))
}

scenario_dag_plot      <- build_dag_plot(c("A", "B", "C", "D"))
scenario_dag_A_plot    <- build_dag_plot("A")
scenario_dag_BCD_plot  <- build_dag_plot(c("B", "C", "D"))

# Annotated 2x2 variant for the parameter-reference supplement: same diagrams,
# edge labels showing default coefficient values (label_val) with per-edge nudges.
build_dag_plot_annot <- function(scenarios_to_show) {
  edges_subset <- edges_all |> filter(scenario %in% scenarios_to_show)
  nodes_subset <- nodes_all |> filter(scenario %in% scenarios_to_show)
  scenario_labels_annot <- c(A = "A. Molecular mediator (primary)",
                             B = "B. Upstream GxE",
                             C = "C. Reverse causation",
                             D = "D. E-M confounding")
  ggplot() +
    geom_segment(data = edges_subset,
                 aes(x = x, y = y, xend = xend, yend = yend,
                     linetype = kind, color = kind),
                 arrow = arrow(length = unit(0.10, "in"), type = "closed"),
                 linewidth = 0.55) +
    geom_text(data = edges_subset,
              aes(x = label_x + nx, y = label_y + ny, label = label_val, color = kind),
              parse = TRUE, size = 3.4, show.legend = FALSE) +
    geom_label(data = nodes_subset,
               aes(x = x, y = y, label = node),
               size = 4.6, fontface = "bold", fill = "white",
               label.padding = unit(0.22, "lines")) +
    facet_wrap(~ scenario, nrow = 2,
               labeller = labeller(scenario = scenario_labels_annot)) +
    scale_linetype_manual(values = c(main = "solid", mod = "dashed")) +
    scale_color_manual(values    = c(main = "grey20", mod = "firebrick")) +
    coord_cartesian(xlim = c(-0.25, 2.25), ylim = c(-0.45, 1.55), clip = "off",
                    expand = FALSE) +
    theme_void(base_size = 13) +
    theme(legend.position = "none",
          strip.text      = element_text(face = "bold", hjust = 0),
          panel.spacing   = unit(0.8, "lines"),
          plot.margin     = margin(6, 6, 6, 6))
}
scenario_dag_annot_plot <- build_dag_plot_annot(c("A", "B", "C", "D"))

scenario_dag_plot

# Setup

In [ ]:
library(patchwork)
# NB: the `mediation` package (used in the supplementary mediated-moderation
# analysis) is called namespace-qualified (mediation::mediate) rather than
# attached, because attaching it loads MASS and masks dplyr::select().
set.seed(42)

N_REPS_POWER <- 100        # replicates per (beta_GxM x ICC) cell
N_REPS_TIE   <- 200        # replicates per TIE scenario (each is a single setting)
N_INDIV      <- 3500       # sample size per replicate
MAF          <- 0.25       # allele frequency (additive coding)
ALPHA        <- 0.05       # significance threshold
BETA_GxM_GRID <- c(0, 0.05, 0.10, 0.15, 0.20)   # true GxM coefficient for power sweep

# --- Measurement-error parameterization via test-retest reliability (ICC) -------
ICC_E_DEFAULT <- 0.5
ICC_M_DEFAULT <- 0.5
ICC_E_GRID    <- c(0.25, 0.5, 0.75, 1.0)
ICC_M_GRID    <- c(0.25, 0.5, 0.75, 1.0)

# Add measurement error to hit a target test-retest reliability (ICC), using the
# draw's own variance: observed = true + N(0, sd(true)^2 * (1/ICC - 1)), so that
# Var(true)/Var(observed) = ICC in expectation. icc >= 1 returns x unchanged.
add_icc_noise <- function(x, icc) {
  if (icc >= 1) return(x)
  x + rnorm(length(x), 0, sd(x) * sqrt(1 / icc - 1))
}

# Pipe-friendly normal-approximation 95% CI helper: adds `lo` and `hi` columns
# to a tibble given a proportion column and the number of trials.
add_asymp_ci <- function(df, prop_col, n) {
  p <- pull(df, {{prop_col}})
  half <- 1.96 * sqrt(p * (1 - p) / n)
  mutate(df, lo = pmax(0, p - half), hi = pmin(1, p + half))
}

---

# Data-generating parameters per scenario

A visual reference for the default data-generating parameters, using the same causal
diagrams as the main figure annotated with their coefficient values.

In [ ]:
scenario_dag_annot_plot

Edge labels give the structural coefficients; all scenarios use N = `r N_INDIV`,
MAF = `r MAF`, and (under the realistic-measurement condition) ICC = 0.5 for E, M,
and U, with `r N_REPS_POWER` (power) and `r N_REPS_TIE` (type I error) replicates per
setting. `beta_GxM` is swept over {0, 0.05, 0.10, 0.15, 0.20} in scenario A. The
per-scenario `beta_GxE` values were chosen to induce an approximately equal *realized*
GxE effect on Y (~0.2): the interaction acts directly on Y in scenarios C and D
(`beta_GxE` = 0.2), but reaches Y only through M in scenario B, so `beta_GxE` = 0.4
compensates for the attenuation by `beta_MY` = 0.5.

# Power under true mediation

In Scenario A, M lies on the causal path between E and Y, and G modifies the `M -> Y`
slope. We generate a *true* exposure `E_true` and *true* metabolite `M_true`, then
construct noisy observed versions by adding Gaussian measurement error scaled to a
target reliability: `E_obs = add_icc_noise(E_true, ICC_E)` (and likewise for M), where
the noise SD is set from each draw's own variance so that `Var(true)/Var(observed) =
ICC`. The GxE and GxM tests are fit on the observed versions, as in real data. We additionally
fit the `G:E`-adjusted GxM test (`Y ~ G*M_obs + G*E_obs`) — the same adjustment we use to
control Type I error in the spurious scenarios — and confirm it leaves power essentially
unchanged under true mediation: the per-replicate GxM z-statistics with and without the
adjustment are near-identical (r ≈ 0.99), so the adjustment that protects against the
spurious scenarios costs nothing when GxM is real. The main figure reports detection
rates at the realistic defaults `(ICC_E = 0.50, ICC_M = 0.50)` and a full sweep of more
ICC values appears in a supplementary figure.

We parameterize measurement error via test-retest reliability (intraclass correlation coefficient, ICC), defined as `Var(true) / Var(observed)`, based on realized empirical variance. Primary scenarios include perfect measurement (all ICC=1) and "realistic" measurement, for which anchors for these values come from:

* Oosterwegel et al. 2023 Environ Sci Technol: LC-HRMS nontargeted serum metabolomics measured ~100 days apart, median ICC = 0.51 (all metabolites) and 0.64 (identified metabolites)
* Kurtze et al. 2008 BMC Med Res Methodol: IPAQ ICC of 0.3 (moderate activity hours), 0.62 (vigorous activity hours), 0.8 (sitting hours)

The exposure-metabolite association is fixed at a reasonably strong value of `β_EM = 0.3` (implied Pearson r ≈ 0.29).

In [ ]:
sim_power <- function(N, maf, beta_GxM,
                      icc_E = 1, icc_M = 1,
                      beta_EM = 0.3, beta_MY = 0.5,
                      sigma_m = 1, sigma_y = 1) {
  G <- rbinom(N, 2, maf)
  E_true <- rnorm(N)
  M_true <- beta_EM * E_true + rnorm(N, 0, sigma_m)
  Y <- beta_MY * M_true + beta_GxM * G * M_true + rnorm(N, 0, sigma_y)
  tibble(
    G     = G,
    E_obs = add_icc_noise(E_true, icc_E),
    M_obs = add_icc_noise(M_true, icc_M),
    Y     = Y
  )
}

run_power_test <- function(df) {
  fit_gxe     <- lm(Y ~ G * E_obs, data = df)
  fit_gxm     <- lm(Y ~ G * M_obs, data = df)
  fit_gxm_adj <- lm(Y ~ G * M_obs + G * E_obs, data = df)   # G:E-adjusted GxM
  list(
    p_gxe     = coef(summary(fit_gxe))["G:E_obs", "Pr(>|t|)"],
    p_gxm     = coef(summary(fit_gxm))["G:M_obs", "Pr(>|t|)"],
    z_gxm     = coef(summary(fit_gxm    ))["G:M_obs", "t value"],
    z_gxm_adj = coef(summary(fit_gxm_adj))["G:M_obs", "t value"]
  )
}

In [ ]:
# Main power simulation: detection rates at the realistic default ICCs.
power_grid <- expand_grid(
  beta_GxM = BETA_GxM_GRID,
  ICC_E    = ICC_E_DEFAULT,
  ICC_M    = ICC_M_DEFAULT,
  rep      = seq_len(N_REPS_POWER)
)

power_results <- power_grid |>
  mutate(.out = pmap(list(beta_GxM, ICC_E, ICC_M), \(b, ie, im) {
    df <- sim_power(N_INDIV, MAF, b, icc_E = ie, icc_M = im)
    run_power_test(df)
  })) |>
  unnest_wider(.out)

In [ ]:
power_summary <- power_results |>
  group_by(beta_GxM, ICC_E, ICC_M) |>
  summarize(
    GxE = mean(p_gxe < ALPHA),
    GxM = mean(p_gxm < ALPHA),
    .groups = "drop"
  )

power_long <- power_summary |>
  pivot_longer(c(GxE, GxM), names_to = "test", values_to = "power") |>
  add_asymp_ci(power, N_REPS_POWER)

In [ ]:
power_plot <- ggplot(power_long, aes(beta_GxM, power, color = test, fill = test,
                                     shape = test, linetype = test)) +
  geom_hline(yintercept = ALPHA, linetype = "dashed", color = "grey50") +
  geom_ribbon(aes(ymin = lo, ymax = hi), alpha = 0.18, color = NA) +
  geom_line(linewidth = 0.6) +
  geom_point(size = 2) +
  scale_color_manual(values = c(GxE = "#d95f02", GxM = "#1b9e77")) +
  scale_fill_manual(values  = c(GxE = "#d95f02", GxM = "#1b9e77")) +
  scale_shape_manual(values = c(GxE = 17, GxM = 16)) +
  scale_linetype_manual(values = c(GxE = "solid", GxM = "solid")) +
  scale_y_continuous(limits = c(0, 1), breaks = seq(0, 1, 0.25)) +
  labs(x = expression(paste("True ", beta[GxM])),
       y = paste0("Power at alpha = ", ALPHA),
       color = NULL, fill = NULL, shape = NULL, linetype = NULL) +
  theme_bw(base_size = 13) +
  theme(plot.title.position = "plot",
        legend.position = "bottom")
power_plot

The `G:E`-adjusted GxM test is the remedy we deploy to control Type I error in the
spurious scenarios below (panels E-G of the combined figure). To show this adjustment
is *free* under true mediation, we compare the per-replicate GxM z-statistic with and
without the `G:E` adjustment at the realistic default ICCs. The two statistics fall
essentially on the identity line, so adjusting for `G:E` neither costs power nor shifts
the test under Scenario A.

In [ ]:
power_z_df <- power_results |> select(beta_GxM, z_gxm, z_gxm_adj)
power_z_r  <- cor(power_z_df$z_gxm, power_z_df$z_gxm_adj)

power_z_plot <- ggplot(power_z_df, aes(z_gxm, z_gxm_adj, color = factor(beta_GxM))) +
  geom_abline(slope = 1, intercept = 0, color = "grey50", linetype = "dashed") +
  geom_point(alpha = 0.5, size = 1.3) +
  scale_color_brewer(palette = "Greens",
                     name = expression(paste("True ", beta[GxM]))) +
  coord_equal() +
  labs(x = "GxM z (unadjusted)", y = "GxM z (G:E-adjusted)") +
  theme_bw(base_size = 13) +
  theme(legend.position = "bottom")
power_z_plot

---

# Comparison with formal mediated-moderation testing

A natural alternative to our G×M screen is a formal *mediated-moderation* test, in
which the G×E interaction (the moderation to be explained) is treated as the
"treatment", the G×M interaction as the "mediator", and the indirect effect tested
with a standard mediation framework. We follow the implementation of Kasela et al.
(2024), who used this approach to ask whether age-specific eQTLs (genotype × age
interactions) are mediated by genotype × cell-type-proportion interactions. The
mediator model regresses the `G×M` product on the `G×E` product (plus main effects),
the outcome model regresses `Y` on both interaction terms (plus main effects), and
the average causal mediation effect (ACME) of the `G×E → G×M → Y` path is estimated
with the `mediation` package. For simulation speed we use the quasi-Bayesian
approximation; Kasela et al. use nonparametric bootstrap (BCa) confidence intervals
for individual analyses.

This formal test is more rigorous than the bare G×M screen — it requires a coherent
indirect path rather than a bare G×M association, and is correspondingly more
specific under the non-mediating scenarios above. Its power, however, runs through
the a-path (`G×E → G×M`), i.e. through the very G×E signal that is *diluted* relative
to the underlying G×M — both by measurement error in E and by the imperfect true
E–M correlation. We therefore quantify how its power and its mediation estimate
degrade as the exposure becomes a weaker or noisier proxy for the operative
metabolite, under true mediation (Scenario A). This is why we adopt the
exposure-independent G×M test (with a G:E adjustment) as the primary screen rather
than the full mediated-moderation model; the formal test remains useful as a
confirmatory sensitivity analysis for prioritized candidates. We further note that
the formal test's proportion-mediated estimate is itself heavily attenuated by
measurement error — under true mediation it falls to roughly 0.3 at ICC = 0.5, and
toward zero as the E–M coupling weakens — so the interpretable effect size it
provides offers little practical advantage over the binary G×M screen.

In [ ]:
# Formal mediated-moderation test (Kasela et al. 2024): the G×E interaction is the
# "treatment", the G×M interaction is the "mediator". Returns the ACME p-value for
# the G×E -> G×M -> Y indirect path.
mediated_moderation_test <- function(df, sims = 400) {
  df$GxE <- df$G * df$E_obs
  df$GxM <- df$G * df$M_obs
  med_model <- lm(GxM ~ G + E_obs + GxE + M_obs, data = df)
  out_model <- lm(Y ~ G + E_obs + GxE + M_obs + GxM, data = df)
  fit <- mediation::mediate(med_model, out_model,
                            treat = "GxE", mediator = "GxM",
                            sims = sims, boot = FALSE)
  fit$d.avg.p   # ACME p-value
}

# Scenario-A generator exposing the E–M coupling beta_EM (mirrors sim_power()).
sim_power_coupling <- function(N, maf, beta_GxM, beta_EM,
                               icc_E = 1, icc_M = 1,
                               beta_MY = 0.5, sigma_m = 1, sigma_y = 1) {
  G <- rbinom(N, 2, maf)
  E_true <- rnorm(N)
  M_true <- beta_EM * E_true + rnorm(N, 0, sigma_m)
  Y <- beta_MY * M_true + beta_GxM * G * M_true + rnorm(N, 0, sigma_y)
  tibble(
    G     = G,
    E_obs = add_icc_noise(E_true, icc_E),
    M_obs = add_icc_noise(M_true, icc_M),
    Y     = Y
  )
}

# Grid: E–M coupling x measurement condition.
BETA_EM_GRID    <- seq(0, 0.3, 0.05)
MEDMOD_BETA_GxM <- 0.15
MEDMOD_ICC      <- c(perfect = 1.0, realistic = 0.5)

# mediate() is the slow step (~1 s/call). Draft mode keeps the knit fast while
# iterating; set MEDMOD_DRAFT <- FALSE for publication-quality curves (the
# cached medmod_run chunk then recomputes once). N_REPS_MEDMOD and MEDMOD_SIMS
# live here so editing them invalidates the medmod_run cache (autodep).
MEDMOD_DRAFT  <- FALSE
N_REPS_MEDMOD <- if (MEDMOD_DRAFT) 10 else 100
MEDMOD_SIMS   <- if (MEDMOD_DRAFT) 150 else 400

In [ ]:
medmod_grid <- expand_grid(
  beta_EM   = BETA_EM_GRID,
  condition = names(MEDMOD_ICC),
  rep       = seq_len(N_REPS_MEDMOD)
)

# The mediate() calls dominate runtime, so fan them out across cores. seed = TRUE
# gives reproducible per-worker RNG (L'Ecuyer streams); user functions are
# auto-exported as globals, and tibble is loaded on workers (sim_power_coupling
# returns a tibble; mediation is called namespace-qualified).
future::plan(future::multisession, workers = max(1, parallel::detectCores() - 1))

medmod_results <- medmod_grid |>
  mutate(.out = furrr::future_pmap(
    list(beta_EM, condition),
    \(bem, cond) {
      icc <- MEDMOD_ICC[[cond]]
      df <- sim_power_coupling(N_INDIV, MAF, MEDMOD_BETA_GxM, bem,
                               icc_E = icc, icc_M = icc)
      list(
        p_gxe = coef(summary(lm(Y ~ G * E_obs, df)))["G:E_obs", "Pr(>|t|)"],
        p_gxm = coef(summary(lm(Y ~ G * M_obs, df)))["G:M_obs", "Pr(>|t|)"],
        p_med = mediated_moderation_test(df, sims = MEDMOD_SIMS)
      )
    },
    .options = furrr::furrr_options(seed = TRUE, packages = c("dplyr", "tibble")))) |>
  unnest_wider(.out)

future::plan(future::sequential)   # release workers

In [ ]:
cond_labels <- c(perfect = "Perfect (ICC = 1)",
                 realistic = "Realistic (ICC = 0.5)")

medmod_power <- medmod_results |>
  group_by(beta_EM, condition) |>
  summarize(
    GxE        = mean(p_gxe < ALPHA),
    GxM        = mean(p_gxm < ALPHA),
    Mediation  = mean(p_med < ALPHA),
    .groups    = "drop"
  ) |>
  pivot_longer(c(GxE, GxM, Mediation), names_to = "test", values_to = "power") |>
  add_asymp_ci(power, N_REPS_MEDMOD) |>
  mutate(condition = factor(condition, levels = names(MEDMOD_ICC),
                            labels = cond_labels[names(MEDMOD_ICC)]),
         test      = factor(test, levels = c("GxE", "GxM", "Mediation")))

In [ ]:
med_cols <- c(GxE = "#d95f02", GxM = "#1b9e77", Mediation = "#7570b3")

panel_power <- ggplot(medmod_power, aes(beta_EM, power, color = test, fill = test,
                                        shape = test)) +
  geom_hline(yintercept = ALPHA, linetype = "dashed", color = "grey50") +
  geom_ribbon(aes(ymin = lo, ymax = hi), alpha = 0.15, color = NA) +
  geom_line(linewidth = 0.6) +
  geom_point(size = 2) +
  facet_wrap(~ condition) +
  scale_color_manual(values = med_cols) +
  scale_fill_manual(values = med_cols) +
  scale_shape_manual(values = c(GxE = 17, GxM = 16, Mediation = 15)) +
  # scale_x_reverse(breaks = BETA_EM_GRID) +
  scale_y_continuous(limits = c(0, 1), breaks = seq(0, 1, 0.25)) +
  labs(x = expression(paste("E-M coupling ", beta[EM])),
       y = paste0("Power at alpha = ", ALPHA),
       color = NULL, fill = NULL, shape = NULL) +
  theme_bw(base_size = 13) +
  theme(legend.position = "bottom", plot.title.position = "plot",
        strip.text = element_text(face = "bold"))

supp_medmod_fig <- panel_power
supp_medmod_fig

---

# Sensitivity: power across the full ICC_E × ICC_M grid

We repeat the Part 1 power sweep across all combinations of `ICC_E_GRID` and
`ICC_M_GRID` to show how detection rate scales with the test-retest reliability of
E and M. The realistic default cell (`ICC_E = ICC_M = 0.5`) sits at the lower end
of the achievable power envelope.

In [ ]:
supp_icc_grid <- expand_grid(
  beta_GxM = BETA_GxM_GRID,
  ICC_E    = ICC_E_GRID,
  ICC_M    = ICC_M_GRID,
  rep      = seq_len(N_REPS_POWER)
)

supp_icc_results <- supp_icc_grid |>
  mutate(.out = pmap(list(beta_GxM, ICC_E, ICC_M), \(b, ie, im) {
    df <- sim_power(N_INDIV, MAF, b, icc_E = ie, icc_M = im)
    run_power_test(df)
  })) |>
  unnest_wider(.out)

supp_icc_summary <- supp_icc_results |>
  group_by(beta_GxM, ICC_E, ICC_M) |>
  summarize(
    GxE = mean(p_gxe < ALPHA),
    GxM = mean(p_gxm < ALPHA),
    .groups = "drop"
  )

supp_icc_long <- supp_icc_summary |>
  pivot_longer(c(GxE, GxM), names_to = "test", values_to = "power") |>
  add_asymp_ci(power, N_REPS_POWER)

In [ ]:
supp_icc_long <- supp_icc_long |>
  mutate(
    ICC_E_label = factor(sprintf("ICC[E] == %.2f", ICC_E),
                         levels = sprintf("ICC[E] == %.2f", ICC_E_GRID)),
    ICC_M_label = factor(sprintf("ICC[M] == %.2f", ICC_M),
                         levels = sprintf("ICC[M] == %.2f", ICC_M_GRID))
  )

supp_icc_fig <- ggplot(supp_icc_long, aes(beta_GxM, power, color = test, fill = test,
                                          shape = test, linetype = test)) +
  geom_hline(yintercept = ALPHA, linetype = "dashed", color = "grey50") +
  geom_ribbon(aes(ymin = lo, ymax = hi), alpha = 0.18, color = NA) +
  geom_line(linewidth = 0.6) +
  geom_point(size = 1.7) +
  facet_grid(ICC_E_label ~ ICC_M_label, labeller = label_parsed) +
  scale_color_manual(values = c(GxE = "#d95f02", GxM = "#1b9e77")) +
  scale_fill_manual(values  = c(GxE = "#d95f02", GxM = "#1b9e77")) +
  scale_shape_manual(values = c(GxE = 17, GxM = 16)) +
  scale_linetype_manual(values = c(GxE = "solid", GxM = "solid")) +
  scale_y_continuous(limits = c(0, 1), breaks = seq(0, 1, 0.25)) +
  labs(x = expression(paste("True ", beta[GxM])),
       y = paste0("Power at alpha = ", ALPHA),
       color = NULL, fill = NULL, shape = NULL, linetype = NULL) +
  theme_bw(base_size = 13) +
  theme(strip.text = element_text(face = "bold"),
        legend.position = "bottom",
        plot.title.position = "plot")
supp_icc_fig

---

# Type I error under non-mediating scenarios

Each scenario carries a true `G x E` effect but no true `G x M` effect. We generate
latent variables according to the specified causal structure, add measurement error
to the observed variables (E and M throughout, plus the confounder U in Scenario D),
and fit the GxM test (`Y ~ G*M`) on the observed variables, mirroring the analysis in
real data. Under the realistic condition every measured variable has ICC = 0.5; under
perfect measurement all are noise-free.

Scenarios B and C share a structural feature that can inflate the GxM test even
though no `G x M` exists: the metabolite carries a marginal association with E whose
slope is modified by G (in B, M tracks E with a G-dependent slope; in C, M is a
downstream marker of an outcome on which `G x E` acts). Fitting `Y ~ G*M` then picks
up a spurious `G:M` term — primarily through G-dependent measurement-error attenuation
of the observed M in B, and additionally through the nonlinear mapping from Y back to
M in C, which inflates the test even under perfect measurement. In both cases the
magnitude is governed by the population-average (marginal) E→M slope and disappears
when that slope is zero. For these two scenarios we therefore report three quantities:
the **baseline** unadjusted test; the same test after **adjusting for `G:E`** (the
remedy available in practice); and a mechanistic check in which the generative model
is altered to have **no marginal E-M correlation** — the E main effect set to zero and
G centered — confirming that the inflation collapses to the nominal level.

In the confounder scenario (D), M has no causal role on Y but shares a common cause U
with E. We compare four GxM tests: unadjusted, plus adjustment for U as a main effect,
for the full `G x U` interaction (U main effect *and* its interaction with G), and for
`G x E`. Because the confounder U is itself measured with error, adjusting for it
reduces but no longer fully eliminates the inflation, illustrating the residual bias
that remains when a confounder is measured imperfectly.

In [ ]:
# Upstream GxE. M tracks E with G-dependent slope; no GxM on Y.
sim_scenario_B <- function(N, maf, beta_GxE,
                           beta_EM = 0.3, beta_MY = 0.5,
                           sigma_m = 1, sigma_y = 1,
                           icc_E = 1, icc_M = 1) {
  G <- rbinom(N, 2, maf); E_true <- rnorm(N)
  M_true <- beta_EM * E_true + beta_GxE * G * E_true + rnorm(N, 0, sigma_m)
  Y <- beta_MY * M_true + rnorm(N, 0, sigma_y)
  tibble(
    G = G,
    E = add_icc_noise(E_true, icc_E),
    M = add_icc_noise(M_true, icc_M),
    Y = Y
  )
}

# Reverse causation. G x E acts on Y directly; M is a downstream marker of Y.
sim_scenario_C <- function(N, maf, beta_GxE,
                           beta_EY = 0.3, beta_YM = 0.5,
                           sigma_y = 1, sigma_m = 1,
                           icc_E = 1, icc_M = 1) {
  G <- rbinom(N, 2, maf); E_true <- rnorm(N)
  Y <- beta_EY * E_true + beta_GxE * G * E_true + rnorm(N, 0, sigma_y)
  M_true <- beta_YM * Y + rnorm(N, 0, sigma_m)
  tibble(
    G = G,
    E = add_icc_noise(E_true, icc_E),
    M = add_icc_noise(M_true, icc_M),
    Y = Y
  )
}

# Confounded null. U is a shared cause of E and M; G x E acts on Y; M has no
# causal role. U is itself measured with error (icc_U < 1).
sim_scenario_D <- function(N, maf, beta_GxE,
                           beta_EY = 0.3, beta_UE = 0.8, beta_UM = 0.7,
                           sigma_e = 1, sigma_m = 1, sigma_y = 1,
                           icc_E = 1, icc_M = 1, icc_U = 1) {
  G <- rbinom(N, 2, maf); U <- rnorm(N)
  E_true <- beta_UE * U + rnorm(N, 0, sigma_e)
  M_true <- beta_UM * U + rnorm(N, 0, sigma_m)
  Y <- beta_EY * E_true + beta_GxE * G * E_true + rnorm(N, 0, sigma_y)
  tibble(
    G = G,
    E = add_icc_noise(E_true, icc_E),
    M = add_icc_noise(M_true, icc_M),
    Y = Y,
    U = add_icc_noise(U, icc_U)
  )
}

# Measurement conditions: realistic ("default") = ICC 0.5 for every measured variable.
ICC_CONDITIONS  <- c("perfect", "default")
ICC_BY_COND     <- c(perfect = 1.0, default = 0.5)
ICC_COND_LABELS <- c(perfect = "Perfect E & M measurement",
                     default = "Realistic measurement error")

Fixed parameter settings, chosen so that each scenario carries a strong motivating
GxE signal on Y (the realized GxE power is recorded in `tie_summary$gxe_power`):

In [ ]:
tie_settings <- list(
  B = list(fn = sim_scenario_B, args = list(beta_GxE = 0.4)),
  C = list(fn = sim_scenario_C, args = list(beta_GxE = 0.2)),
  D = list(fn = sim_scenario_D, args = list(beta_GxE = 0.2))
)

In [ ]:
run_tie_one <- function(scenario, df) {
  fit_unadj  <- lm(Y ~ G * M, data = df)
  fit_gxe    <- lm(Y ~ G * E, data = df)
  fit_adj_GxE <- lm(Y ~ G * M + G * E, data = df)   # G:E adjustment: applies to all scenarios
  out <- list(
    p_unadj      = coef(summary(fit_unadj))["G:M", "Pr(>|t|)"],
    p_adj_U_main = NA_real_,
    p_adj_GxU    = NA_real_,
    p_adj_GxE    = coef(summary(fit_adj_GxE))["G:M", "Pr(>|t|)"],
    p_gxe        = coef(summary(fit_gxe  ))["G:E", "Pr(>|t|)"]
  )
  if (scenario == "D") {   # U-based adjustments only meaningful where U exists
    fit_adj_U_main <- lm(Y ~ G * M + U,     data = df)
    fit_adj_GxU    <- lm(Y ~ G * M + G * U, data = df)
    out$p_adj_U_main <- coef(summary(fit_adj_U_main))["G:M", "Pr(>|t|)"]
    out$p_adj_GxU    <- coef(summary(fit_adj_GxU   ))["G:M", "Pr(>|t|)"]
  }
  out
}

tie_grid <- expand_grid(
  scenario      = names(tie_settings),
  icc_condition = ICC_CONDITIONS,
  rep           = seq_len(N_REPS_TIE)
)

t0 <- Sys.time()
tie_results <- tie_grid |>
  mutate(.out = pmap(list(scenario, icc_condition), \(sc, cond) {
    cfg <- tie_settings[[sc]]
    icc <- ICC_BY_COND[[cond]]
    # Scenario D additionally measures the confounder U with error.
    extra <- if (sc == "D") list(icc_U = icc) else list()
    df  <- do.call(cfg$fn, c(list(N = N_INDIV, maf = MAF),
                             cfg$args,
                             list(icc_E = icc, icc_M = icc),
                             extra))
    run_tie_one(sc, df)
  })) |>
  unnest_wider(.out)
message(sprintf("TIE sims: %d fits in %.1f s",
                nrow(tie_grid),
                as.numeric(Sys.time() - t0, units = "secs")))

tie_summary <- tie_results |>
  group_by(scenario, icc_condition) |>
  summarize(
    tie_unadj      = mean(p_unadj      < ALPHA),
    tie_adj_U_main = mean(p_adj_U_main < ALPHA),  # NaN for B and C
    tie_adj_GxU    = mean(p_adj_GxU    < ALPHA),  # NaN for B and C
    tie_adj_GxE    = mean(p_adj_GxE    < ALPHA),  # NaN for B and C
    gxe_power      = mean(p_gxe        < ALPHA),
    n_adj          = sum(!is.na(p_adj_GxU)),
    .groups        = "drop"
  )

In [ ]:
pd_tie <- position_dodge(width = 0.7)

# Scenario-B variants: baseline, baseline + G:E adjustment, and a no-marginal-E-M
# generative model (beta_EM = 0, G centered). center_G_in_dgp shifts G in the DGP.
sim_B_variant <- function(N, maf, beta_EM, beta_GxE = 0.4,
                          beta_MY = 0.5, center_G_in_dgp = FALSE,
                          sigma_m = 1, sigma_y = 1,
                          icc_E = 1, icc_M = 1) {
  G <- rbinom(N, 2, maf); E_true <- rnorm(N)
  G_in_eq <- if (center_G_in_dgp) G - 2 * maf else G
  M_true <- beta_EM * E_true + beta_GxE * G_in_eq * E_true + rnorm(N, 0, sigma_m)
  Y <- beta_MY * M_true + rnorm(N, 0, sigma_y)
  tibble(
    G = G,
    E = add_icc_noise(E_true, icc_E),
    M = add_icc_noise(M_true, icc_M),
    Y = Y
  )
}

B_VARIANT_LEVELS <- c("full", "full_adj_GxE", "no_marg")
b_variant_labels <- c(
  full              = "Baseline",
  full_adj_GxE      = "Baseline +\nG:E adjustment",
  no_marg           = "No marginal\nE-M correlation"
)

b_variant_dgp_args <- list(
  full              = list(beta_EM = 0.3),
  full_adj_GxE      = list(beta_EM = 0.3),
  no_marg           = list(beta_EM = 0.0, center_G_in_dgp = TRUE)
)

b_grid <- expand_grid(
  variant       = B_VARIANT_LEVELS,
  icc_condition = ICC_CONDITIONS,
  rep           = seq_len(N_REPS_TIE)
)
b_results <- b_grid |>
  mutate(.out = pmap(list(variant, icc_condition), \(v, cond) {
    icc  <- ICC_BY_COND[[cond]]
    args <- c(list(N = N_INDIV, maf = MAF),
              b_variant_dgp_args[[v]],
              list(icc_E = icc, icc_M = icc))
    df   <- do.call(sim_B_variant, args)
    if (v == "full_adj_GxE") {
      list(p = coef(summary(lm(Y ~ G * M + G * E, df)))["G:M", "Pr(>|t|)"])
    } else {
      list(p = coef(summary(lm(Y ~ G * M, df)))["G:M", "Pr(>|t|)"])
    }
  })) |>
  unnest_wider(.out)

tie_b_dt <- b_results |>
  group_by(variant, icc_condition) |>
  summarize(tie = mean(p < ALPHA), .groups = "drop") |>
  mutate(
    model         = factor(variant, levels = B_VARIANT_LEVELS,
                           labels = b_variant_labels[B_VARIANT_LEVELS]),
    icc_condition = factor(icc_condition, levels = ICC_CONDITIONS,
                           labels = ICC_COND_LABELS[ICC_CONDITIONS])
  ) |>
  add_asymp_ci(tie, N_REPS_TIE)

tie_b_plot <- ggplot(tie_b_dt, aes(model, tie, alpha = icc_condition)) +
  geom_col(fill = "#7570b3", color = "grey20", width = 0.7,
           position = pd_tie) +
  geom_errorbar(aes(ymin = lo, ymax = hi), width = 0.18, position = pd_tie) +
  geom_hline(yintercept = ALPHA, linetype = "dashed", color = "grey40") +
  scale_alpha_manual(values = setNames(c(0.45, 1.0), ICC_COND_LABELS[ICC_CONDITIONS])) +
  scale_y_continuous(limits = c(0, 1), breaks = seq(0, 1, 0.2)) +
  labs(x = NULL, y = "Type I error rate of GxM test",
       alpha = NULL,
       title = "Upstream GxE") +
  theme_bw(base_size = 13) +
  theme(plot.title.position = "plot",
        axis.text.x = element_text(size = 9),
        legend.position = "bottom",
        legend.direction = "vertical")

In [ ]:
# Scenario-C variants: baseline, baseline + G:E adjustment, and a no-marginal-E-M
# generative model (beta_EY = 0, G centered). center_G_in_dgp shifts G in the DGP.
sim_C_variant <- function(N, maf, beta_EY, beta_GxE = 0.2,
                          beta_YM = 0.5, center_G_in_dgp = FALSE,
                          sigma_y = 1, sigma_m = 1,
                          icc_E = 1, icc_M = 1) {
  G <- rbinom(N, 2, maf); E_true <- rnorm(N)
  G_in_eq <- if (center_G_in_dgp) G - 2 * maf else G
  Y <- beta_EY * E_true + beta_GxE * G_in_eq * E_true + rnorm(N, 0, sigma_y)
  M_true <- beta_YM * Y + rnorm(N, 0, sigma_m)
  tibble(
    G = G,
    E = add_icc_noise(E_true, icc_E),
    M = add_icc_noise(M_true, icc_M),
    Y = Y
  )
}

C_VARIANT_LEVELS <- c("full", "full_adj_GxE", "no_marg")
c_variant_labels <- c(
  full              = "Baseline",
  no_marg           = "No marginal\nE-M correlation",
  full_adj_GxE      = "Baseline +\nG:E adjustment"
)

c_variant_dgp_args <- list(
  full              = list(beta_EY = 0.3),
  no_marg           = list(beta_EY = 0.0, center_G_in_dgp = TRUE),
  full_adj_GxE      = list(beta_EY = 0.3)
)

c_grid <- expand_grid(
  variant       = C_VARIANT_LEVELS,
  icc_condition = ICC_CONDITIONS,
  rep           = seq_len(N_REPS_TIE)
)
c_results <- c_grid |>
  mutate(.out = pmap(list(variant, icc_condition), \(v, cond) {
    icc  <- ICC_BY_COND[[cond]]
    args <- c(list(N = N_INDIV, maf = MAF),
              c_variant_dgp_args[[v]],
              list(icc_E = icc, icc_M = icc))
    df   <- do.call(sim_C_variant, args)
    if (v == "full_adj_GxE") {
      list(p = coef(summary(lm(Y ~ G * M + G * E, df)))["G:M", "Pr(>|t|)"])
    } else {
      list(p = coef(summary(lm(Y ~ G * M, df)))["G:M", "Pr(>|t|)"])
    }
  })) |>
  unnest_wider(.out)

tie_c_dt <- c_results |>
  group_by(variant, icc_condition) |>
  summarize(tie = mean(p < ALPHA), .groups = "drop") |>
  mutate(
    model         = factor(variant, levels = C_VARIANT_LEVELS,
                           labels = c_variant_labels[C_VARIANT_LEVELS]),
    icc_condition = factor(icc_condition, levels = ICC_CONDITIONS,
                           labels = ICC_COND_LABELS[ICC_CONDITIONS])
  ) |>
  add_asymp_ci(tie, N_REPS_TIE)

tie_c_variants_plot <- ggplot(tie_c_dt, aes(model, tie, alpha = icc_condition)) +
  geom_col(fill = "#d95f02", color = "grey20", width = 0.7,
           position = pd_tie) +
  geom_errorbar(aes(ymin = lo, ymax = hi), width = 0.18, position = pd_tie) +
  geom_hline(yintercept = ALPHA, linetype = "dashed", color = "grey40") +
  scale_alpha_manual(values = setNames(c(0.45, 1.0), ICC_COND_LABELS[ICC_CONDITIONS])) +
  scale_y_continuous(limits = c(0, 1), breaks = seq(0, 1, 0.2)) +
  labs(x = NULL, y = "Type I error rate of GxM test",
       alpha = NULL,
       title = "Reverse causation") +
  theme_bw(base_size = 13) +
  theme(plot.title.position = "plot",
        axis.text.x = element_text(size = 9),
        legend.position = "bottom",
        legend.direction = "vertical")

In [ ]:
D_MODEL_LEVELS <- c("Unadjusted",
                    "U-adjusted",
                    "G:U-adjusted",
                    "G:E-adjusted")

# Panel G also measures U with error, so its "perfect" condition covers E, M, and U.
ICC_COND_LABELS_D <- c(perfect = "Perfect E,M,U measurement",
                       default = "Realistic measurement error")

tie_d_dt <- tie_summary |>
  filter(scenario == "D") |>
  select(icc_condition, tie_unadj, tie_adj_U_main, tie_adj_GxU, tie_adj_GxE) |>
  pivot_longer(starts_with("tie_"), names_to = "model_key", values_to = "tie") |>
  mutate(
    model = factor(
      recode(model_key,
             tie_unadj      = D_MODEL_LEVELS[1],
             tie_adj_U_main = D_MODEL_LEVELS[2],
             tie_adj_GxU    = D_MODEL_LEVELS[3],
             tie_adj_GxE    = D_MODEL_LEVELS[4]),
      levels = D_MODEL_LEVELS
    ),
    icc_condition = factor(icc_condition, levels = ICC_CONDITIONS,
                           labels = ICC_COND_LABELS_D[ICC_CONDITIONS])
  ) |>
  select(-model_key) |>
  add_asymp_ci(tie, N_REPS_TIE)

tie_d_variants_plot <- ggplot(tie_d_dt, aes(model, tie, alpha = icc_condition)) +
  geom_col(fill = "#1b9e77", color = "grey20", width = 0.7,
           position = pd_tie) +
  geom_errorbar(aes(ymin = lo, ymax = hi), width = 0.18, position = pd_tie) +
  geom_hline(yintercept = ALPHA, linetype = "dashed", color = "grey40") +
  scale_alpha_manual(values = setNames(c(0.45, 1.0), ICC_COND_LABELS_D[ICC_CONDITIONS])) +
  scale_y_continuous(limits = c(0, 1), breaks = seq(0, 1, 0.2)) +
  labs(x = NULL, y = "Type I error rate of GxM test",
       alpha = NULL,
       title = "E-M confounder") +
  theme_bw(base_size = 13) +
  theme(plot.title.position = "plot",
        axis.text.x = element_text(size = 10),
        legend.position = "bottom",
        legend.direction = "vertical")

---

# Sensitivity: type I error under a nonnegative (gamma-distributed) E

The main analyses use `E ~ N(0, 1)`. Many real exposures are nonnegative (e.g.,
self-reported MVPA), so as a sensitivity analysis we replace the standard-normal E
with a Gamma(shape = 4, rate = 2) draw — same variance (1), but mean = 2 and a
right-skewed, strictly nonnegative support. We rerun the headline TIE simulations
(Scenarios B, C, D at the same fixed parameter settings) under this alternative E
distribution.

In [ ]:
# Gamma(shape = 4, rate = 2) has mean = 2, var = 1, support [0, Inf).
rgamma_e <- function(N) rgamma(N, shape = 4, rate = 2)

In [ ]:
# Sim functions that draw E from a user-supplied generator (gamma sensitivity);
# these use perfect measurement (no added noise).
sim_B_supp <- function(N, maf, beta_GxE, e_gen,
                       beta_EM = 0.3, beta_MY = 0.5,
                       sigma_m = 1, sigma_y = 1) {
  G <- rbinom(N, 2, maf); E <- e_gen(N)
  M <- beta_EM * E + beta_GxE * G * E + rnorm(N, 0, sigma_m)
  Y <- beta_MY * M + rnorm(N, 0, sigma_y)
  tibble(G = G, E = E, M = M, Y = Y)
}
sim_C_supp <- function(N, maf, beta_GxE, e_gen,
                       beta_EY = 0.3, beta_YM = 0.5,
                       sigma_y = 1, sigma_m = 1) {
  G <- rbinom(N, 2, maf); E <- e_gen(N)
  Y <- beta_EY * E + beta_GxE * G * E + rnorm(N, 0, sigma_y)
  M <- beta_YM * Y + rnorm(N, 0, sigma_m)
  tibble(G = G, E = E, M = M, Y = Y)
}
sim_D_supp <- function(N, maf, beta_GxE, e_gen,
                       beta_EY = 0.3, beta_UE = 0.8, beta_UM = 0.7,
                       sigma_e = 1, sigma_m = 1, sigma_y = 1) {
  G <- rbinom(N, 2, maf); U <- rnorm(N)
  # Gamma E plus a U-driven shift: preserves the E-M confounding via U.
  E <- e_gen(N) + beta_UE * U
  M <- beta_UM * U + rnorm(N, 0, sigma_m)
  Y <- beta_EY * E + beta_GxE * G * E + rnorm(N, 0, sigma_y)
  tibble(G = G, E = E, M = M, Y = Y, U = U)
}

In [ ]:
# For each replicate, fit all candidate models for the scenario and return p-values.
# Fixed-length named list so unnest_wider creates a consistent set of columns.
supp_one_rep <- function(scenario) {
  df <- switch(scenario,
    B = sim_B_supp(N_INDIV, MAF, beta_GxE = 0.4, e_gen = rgamma_e),
    C = sim_C_supp(N_INDIV, MAF, beta_GxE = 0.2, e_gen = rgamma_e),
    D = sim_D_supp(N_INDIV, MAF, beta_GxE = 0.2, e_gen = rgamma_e)
  )
  out <- list(unadj  = coef(summary(lm(Y ~ G * M, df)))["G:M", "Pr(>|t|)"],
              adjGxE = coef(summary(lm(Y ~ G * M + G * E, df)))["G:M", "Pr(>|t|)"],
              adjU   = NA_real_,
              adjGxU = NA_real_)
  if (scenario == "D") {
    out$adjU   <- coef(summary(lm(Y ~ G * M + U,     df)))["G:M", "Pr(>|t|)"]
    out$adjGxU <- coef(summary(lm(Y ~ G * M + G * U, df)))["G:M", "Pr(>|t|)"]
  }
  out
}

supp_results <- expand_grid(
  scenario = c("B", "C", "D"),
  rep      = seq_len(N_REPS_TIE)
) |>
  mutate(.out = map(scenario, supp_one_rep)) |>
  unnest_wider(.out)

supp_summary <- supp_results |>
  group_by(scenario) |>
  summarize(
    tie_unadj      = mean(unadj  < ALPHA, na.rm = TRUE),
    tie_adj_U_main = mean(adjU   < ALPHA, na.rm = TRUE),
    tie_adj_GxU    = mean(adjGxU < ALPHA, na.rm = TRUE),
    tie_adj_GxE    = mean(adjGxE < ALPHA, na.rm = TRUE),
    .groups        = "drop"
  )

knitr::kable(supp_summary, digits = 3,
             caption = "Supplementary: TIE under Gamma(4, 2) exposure E.")

In [ ]:
tie_perf <- tie_summary |> filter(icc_condition == "perfect")
pull_tie <- function(df, sc, col) df |> filter(scenario == sc) |> pull({{col}})

compare_tbl <- tibble(
  scenario = c("B", "B adj G:E", "C", "C adj G:E",
               "D unadj", "D adj U", "D adj G:U", "D adj G:E"),
  normal_E = c(
    pull_tie(tie_perf,    "B", tie_unadj),
    pull_tie(tie_perf,    "B", tie_adj_GxE),
    pull_tie(tie_perf,    "C", tie_unadj),
    pull_tie(tie_perf,    "C", tie_adj_GxE),
    pull_tie(tie_perf,    "D", tie_unadj),
    pull_tie(tie_perf,    "D", tie_adj_U_main),
    pull_tie(tie_perf,    "D", tie_adj_GxU),
    pull_tie(tie_perf,    "D", tie_adj_GxE)
  ),
  gamma_E = c(
    pull_tie(supp_summary, "B", tie_unadj),
    pull_tie(supp_summary, "B", tie_adj_GxE),
    pull_tie(supp_summary, "C", tie_unadj),
    pull_tie(supp_summary, "C", tie_adj_GxE),
    pull_tie(supp_summary, "D", tie_unadj),
    pull_tie(supp_summary, "D", tie_adj_U_main),
    pull_tie(supp_summary, "D", tie_adj_GxU),
    pull_tie(supp_summary, "D", tie_adj_GxE)
  )
)
knitr::kable(compare_tbl, digits = 3,
             col.names = c("Setting", "E ~ N(0,1)", "E ~ Gamma(4,2)"),
             caption = "TIE comparison: normal vs gamma-distributed E.")

In [ ]:
supp_long <- compare_tbl |>
  pivot_longer(-scenario, names_to = "e_dist", values_to = "tie") |>
  mutate(
    e_dist   = factor(e_dist, levels = c("normal_E", "gamma_E"),
                      labels = c("E ~ N(0, 1)", "E ~ Gamma(4, 2)")),
    scenario = factor(scenario, levels = compare_tbl$scenario)
  )

supp_fig <- ggplot(supp_long, aes(scenario, tie, fill = e_dist)) +
  geom_col(position = position_dodge(width = 0.75), width = 0.65, color = "grey20") +
  geom_hline(yintercept = ALPHA, linetype = "dashed", color = "grey40") +
  scale_fill_manual(values = c("E ~ N(0, 1)" = "#7570b3",
                               "E ~ Gamma(4, 2)" = "#d95f02")) +
  scale_y_continuous(limits = c(0, 1), breaks = seq(0, 1, 0.2)) +
  labs(x = NULL, y = "Type I error rate of GxM test",
       fill = "Exposure distribution") +
  theme_bw(base_size = 13) +
  theme(legend.position = "bottom",
        plot.title.position = "plot",
        axis.text.x = element_text(size = 9, angle = 30, hjust = 1))
supp_fig

---

# Combined manuscript figure

Three-row layout: the primary-model DAG with its power plot and the `G:E`-adjustment
equivalence panel (Row 1), the three spurious-scenario DAGs (Row 2), and the
corresponding TIE bar plots one per scenario (Row 3). Panel tags A-G are assigned in
reading order.

In [ ]:
primary_row  <- scenario_dag_A_plot + power_plot + power_z_plot +
  plot_layout(widths = c(1, 1.4, 1.2))

spurious_dags  <- scenario_dag_BCD_plot

tie_row <- tie_b_plot + tie_c_variants_plot + tie_d_variants_plot +
  plot_layout(widths = c(1, 1, 1))

combined_fig <-
  primary_row /
  spurious_dags /
  tie_row +
  plot_layout(heights = c(1.0, 0.8, 1.3)) +
  plot_annotation(tag_levels = "A") &
  theme(plot.tag = element_text(face = "bold", size = 16))
combined_fig

---

# Reproducibility

In [ ]:
sessionInfo()